# **Задание №9. Разработка стратегии тестирования и тест-кейсов для информационной системы**

# СТРАТЕГИЯ ТЕСТИРОВАНИЯ И ТЕСТ-КЕЙСЫ

## 1. СТРАТЕГИЯ ТЕСТИРОВАНИЯ

### 1.1. Цели и задачи

Обеспечить соответствие платформы сегментации ДЗЗ на основе HyenaPixel функциональным и нефункциональным требованиям: корректность работы модулей импорта сцен, обучения, инференса, управления данными и безопасности. Выявить дефекты до развёртывания в продуктивной среде, подтвердить достижение целевых метрик качества модели (mIoU ≥ 0.75) и производительности инференса.

Область применения: Платформа сегментации ДЗЗ на основе HyenaPixel, версия 1.0.

### 1.2. Область тестирования

Включено:

    Модуль аутентификации и управления пользователями

    Модуль импорта и управления сценами ДЗЗ

    Модуль формирования датасетов и разметки

    Модуль самоконтролируемого предобучения (SSL)

    Модуль обучения модели сегментации

    Модуль инференса и визуализации результатов

    REST API всех модулей

    Безопасность: защита данных, авторизация по ролям

    Производительность: время инференса, время загрузки сцен

Исключено:

    Тестирование сторонних хранилищ данных (S3/HTTP)

    Тестирование GPU-драйверов и низкоуровневого ПО

    Тестирование операционной системы и контейнерной инфраструктуры

### 1.3. Виды тестирования

Модульное (Unit) — тестирование отдельных функций: препроцессинга тайлов, расчёта IoU, логики SSL-задач

Интеграционное — взаимодействие модулей: DataService ↔ TrainingService, InferenceEngine ↔ ModelRegistry

Функциональное (Black Box) — проверка пользовательских сценариев через UI и REST API

Нефункциональное — производительность инференса, безопасность, юзабилити

Регрессионное — проверка стабильности после внесения изменений

Smoke-тестирование — быстрая проверка ключевой функциональности перед полным циклом

### 1.4. Инструменты тестирования

| Тип тестирования | Инструмент        | Обоснование                                                      |
| ---------------- | ----------------- | ---------------------------------------------------------------- |
| Модульное        | pytest            | Стандарт для Python-проектов; поддержка фикстур и параметризации |
| Интеграционное   | pytest + requests | Тестирование API-взаимодействия между сервисами                  |
| Нагрузочное      | Locust            | Python-native, гибкое создание сценариев нагрузки                |
| Безопасность     | OWASP ZAP         | Автоматическое сканирование уязвимостей REST API                 |
| Покрытие кода    | pytest-cov        | Генерация отчётов об охвате кода тестами                         |
| CI/CD-интеграция | GitHub Actions    | Автозапуск тестов при коммитах                                   |

### 1.5. Критерии входа и выхода

Entry Criteria:

    Код зафиксирован в системе контроля версий (Git)

    Тестовый стенд развёрнут и доступен

    Тестовые датасеты подготовлены (минимум 3 сцены с разметкой)

    Документация API актуальна

Exit Criteria:

    Все тест-кейсы с приоритетом «Критический» выполнены и имеют статус Passed

    Pass Rate ≥ 98% для критических тестов

    Покрытие критических и высокоприоритетных требований: 100%

    Отсутствуют открытые дефекты уровня Critical и Major

    Метрика модели mIoU ≥ 0.75 подтверждена на тестовом наборе

### 1.6. Метрики качества

Покрытие требований «Критический» и «Высокий»: ≥ 100%

Покрытие кода unit-тестами: ≥ 75%

Pass Rate перед релизом: ≥ 98%

DDE (Defect Detection Efficiency): ≥ 90%

Время инференса на одну сцену (1024×1024 пикселей): ≤ 5 сек

## 2. ФУНКЦИОНАЛЬНЫЕ ТЕСТ-КЕЙСЫ

### 2.1. Модуль: AUTH — Аутентификация и управление пользователями


TC-AUTH-001: Успешная авторизация с валидными учётными данными
| Поле                 | Описание                                                       |
| -------------------- | -------------------------------------------------------------- |
| ID                   | TC-AUTH-001                                                    |
| Название             | Успешная авторизация пользователя с корректным email и паролем |
| Приоритет            | Критический                                                    |
| Тип                  | Функциональное, позитивное                                     |
| Связь с требованиями | ФТ-AUTH-001: Авторизация пользователя                          |

Предусловия:

    В базе данных существует пользователь: ml_engineer@test.ru, роль ML_ENGINEER

    Сервис аутентификации запущен

**Тестовые данные:**
```
email: ml_engineer@test.ru
password: TestPass_2025!
```
**Шаги выполнения:**

| № | Действие                                                      | Ожидаемый результат                   |
| - | ------------------------------------------------------------- | ------------------------------------- |
| 1 | Отправить POST /api/auth/login с корректными учётными данными | Запрос принят сервером                |
| 2 | Проверить HTTP-статус ответа                                  | Статус 200 OK                         |
| 3 | Проверить тело ответа                                         | Ответ содержит access_token и user_id |
| 4 | Использовать полученный токен для запроса GET /api/scenes     | Статус 200, возвращается список сцен  |

**Постусловия:**
Пользователь аутентифицирован, сессионный токен активен

TC-AUTH-002: Авторизация с неверным паролем

| Поле                 | Описание                                                       |
| -------------------- | -------------------------------------------------------------- |
| ID                   | TC-AUTH-002                                                    |
| Название             | Отклонение авторизации при неверном пароле                     |
| Приоритет            | Критический                                                    |
| Тип                  | Функциональное, негативное                                     |
| Связь с требованиями | ФТ-AUTH-001, НФТ-Б-001: Защита от несанкционированного доступа |

Предусловия:

    Пользователь ml_engineer@test.ru существует в базе


Тестовые данные:

    email: ml_engineer@test.ru
    password: WrongPassword123

Шаги выполнения:
| № | Действие                                          | Ожидаемый результат                         |
| - | ------------------------------------------------- | ------------------------------------------- |
| 1 | Отправить POST /api/auth/login с неверным паролем | Запрос отправлен                            |
| 2 | Проверить HTTP-статус ответа                      | Статус 401 Unauthorized                     |
| 3 | Проверить тело ответа                             | Сообщение: «Неверный email или пароль»      |
| 4 | Попытаться использовать несуществующий токен      | Статус 403 Forbidden для любого API-запроса |

Постусловия:
```
    Токен доступа НЕ выдан, сессия НЕ создана
```


TC-AUTH-003: Доступ к ресурсу без токена

| Поле                 | Описание                                                          |
| -------------------- | ----------------------------------------------------------------- |
| ID                   | TC-AUTH-003                                                       |
| Название             | Запрет доступа к API без токена авторизации                       |
| Приоритет            | Критический                                                       |
| Тип                  | Функциональное, негативное, безопасность                          |
| Связь с требованиями | НФТ-Б-001: Авторизация обязательна для всех защищённых эндпоинтов |

Предусловия:

    Токен авторизации отсутствует в заголовке запроса

Шаги выполнения:

| № | Действие                                              | Ожидаемый результат     |
| - | ----------------------------------------------------- | ----------------------- |
| 1 | Отправить GET /api/scenes без заголовка Authorization | Статус 401 Unauthorized |
| 2 | Отправить POST /api/experiments без токена            | Статус 401 Unauthorized |
| 3 | Отправить GET /api/inference/runs без токена          | Статус 401 Unauthorized |

Постусловия:

    Данные не возвращены, доступ заблокирован



TC-AUTH-004: Ограничение доступа по роли (RBAC)

| Поле                 | Описание                                                    |
| -------------------- | ----------------------------------------------------------- |
| ID                   | TC-AUTH-004                                                 |
| Название             | Пользователь с ролью RS_ANALYST не может запустить обучение |
| Приоритет            | Высокий                                                     |
| Тип                  | Функциональное, негативное, авторизация                     |
| Связь с требованиями | НФТ-Б-002: Разграничение доступа по ролям                   |

Предусловия:

    Пользователь analyst@test.ru авторизован с ролью RS_ANALYST

    Токен получен

Шаги выполнения:

| № | Действие                                            | Ожидаемый результат                                    |
| - | --------------------------------------------------- | ------------------------------------------------------ |
| 1 | Отправить POST /api/experiments с токеном аналитика | Статус 403 Forbidden                                   |
| 2 | Проверить тело ответа                               | Сообщение: «Недостаточно прав для выполнения операции» |
| 3 | Отправить GET /api/scenes с тем же токеном          | Статус 200 OK (сцены доступны аналитику)               |

2.2. Модуль: DATA — Импорт и управление сценами ДЗЗ

TC-DATA-001: Успешный импорт сцены ДЗЗ

| Поле                 | Описание                                            |
| -------------------- | --------------------------------------------------- |
| ID                   | TC-DATA-001                                         |
| Название             | Импорт валидной сцены ДЗЗ с корректными метаданными |
| Приоритет            | Критический                                         |
| Тип                  | Функциональное, позитивное                          |
| Связь с требованиями | ФТ-DATA-001: Импорт сцен ДЗЗ                        |

Предусловия:

    Пользователь авторизован как ML_ENGINEER

    Подготовлен тестовый файл сцены (GeoTIFF, 512×512 пикселей, 4 спектральных канала, 10 м/пкс)

Тестовые данные:
```
Файл: test_scene_sentinel2.tif
Источник: Sentinel-2, 15.05.2024
Размер: 512×512, 4 band, 10.0 м/пкс

Шаги выполнения:

| № | Действие                                                    | Ожидаемый результат                                     |
| - | ----------------------------------------------------------- | ------------------------------------------------------- |
| 1 | Отправить POST /api/scenes с multipart-файлом и метаданными | Статус 201 Created                                      |
| 2 | Проверить тело ответа                                       | Возвращён scene_id, поля width=512, height=512, bands=4 |
| 3 | Отправить GET /api/scenes/{scene_id}                        | Статус 200, метаданные соответствуют загруженным        |
| 4 | Проверить наличие файла в хранилище                         | Файл доступен по URI из ответа                          |

Постусловия:

    Сцена сохранена в базе данных с уникальным ID

    Файл доступен через storage_uri

TC-DATA-002: Импорт сцены с неподдерживаемым форматом файла

| Поле                 | Описание                                                  |
| -------------------- | --------------------------------------------------------- |
| ID                   | TC-DATA-002                                               |
| Название             | Отклонение импорта файла неподдерживаемого формата (JPEG) |
| Приоритет            | Высокий                                                   |
| Тип                  | Функциональное, негативное, валидация                     |
| Связь с требованиями | ФТ-DATA-001: Поддерживаемые форматы GeoTIFF, PNG          |

Предусловия:

    Пользователь авторизован

Тестовые данные:
```
Файл: photo.jpg (обычная фотография JPEG)

Шаги выполнения:

| № | Действие                                 | Ожидаемый результат                                                          |
| - | ---------------------------------------- | ---------------------------------------------------------------------------- |
| 1 | Отправить POST /api/scenes с файлом .jpg | Статус 400 Bad Request                                                       |
| 2 | Проверить тело ответа                    | Сообщение: «Неподдерживаемый формат файла. Допустимые форматы: GeoTIFF, PNG» |
| 3 | Проверить, что сцена не создана в БД     | GET /api/scenes — новая запись отсутствует                                   |

TC-DATA-003: Загрузка разметки для существующей сцены

| Поле                 | Описание                                    |
| -------------------- | ------------------------------------------- |
| ID                   | TC-DATA-003                                 |
| Название             | Успешная привязка маски сегментации к сцене |
| Приоритет            | Высокий                                     |
| Тип                  | Функциональное, позитивное                  |
| Связь с требованиями | ФТ-DATA-002: Загрузка разметки сцен         |

Предусловия:

    Сцена с scene_id=42 существует в системе

    Подготовлен файл маски (PNG, 512×512 пикселей, классы: 0=фон, 1=вода, 2=здания)

Шаги выполнения:

| № | Действие                                                | Ожидаемый результат                                              |
| - | ------------------------------------------------------- | ---------------------------------------------------------------- |
| 1 | Отправить POST /api/labels с scene_id=42 и файлом маски | Статус 201 Created                                               |
| 2 | Получить GET /api/labels?scene_id=42                    | Возвращена запись разметки с label_id, mask_path, schema_version |
| 3 | Попытаться загрузить вторую маску для той же сцены      | Статус 409 Conflict: «Разметка для данной сцены уже существует»  |

2.3. Модуль: DATASET — Формирование датасетов

TC-DATASET-001: Создание датасета с разбиением train/val/test

| Поле                 | Описание                                                      |
| -------------------- | ------------------------------------------------------------- |
| ID                   | TC-DATASET-001                                                |
| Название             | Создание обучающего датасета с разбиением сцен по подвыборкам |
| Приоритет            | Критический                                                   |
| Тип                  | Функциональное, позитивное                                    |
| Связь с требованиями | ФТ-DS-001: Формирование обучающего датасета                   |v

Предусловия:

    Существуют 10 сцен с разметкой (scene_id: 1–10)

    Пользователь авторизован как ML_ENGINEER

Тестовые данные:

```
Название датасета: "Sentinel2_Urban_v1"
Тип: LABELED
Сцены: [1..10]
Разбиение: train=7, val=2, test=1
```
Шаги выполнения:

| № | Действие                                       | Ожидаемый результат                                       |
| - | ---------------------------------------------- | --------------------------------------------------------- |
| 1 | Отправить POST /api/datasets с параметрами     | Статус 201 Created, возвращён dataset_id                  |
| 2 | Получить GET /api/datasets/{dataset_id}/scenes | Список из 10 сцен с полем split                           |
| 3 | Проверить разбиение                            | Ровно 7 сцен с split=train, 2 с val, 1 с test             |
| 4 | Попытаться создать датасет без сцен            | Статус 400: «Датасет должен содержать хотя бы одну сцену» |

TC-DATASET-002: Повторное использование сцены в двух датасетах

| Поле                 | Описание                                             |
| -------------------- | ---------------------------------------------------- |
| ID                   | TC-DATASET-002                                       |
| Название             | Одна сцена входит в несколько датасетов одновременно |
| Приоритет            | Средний                                              |
| Тип                  | Функциональное, позитивное                           |
| Связь с требованиями | ФТ-DS-001: Связь N:M между датасетами и сценами      |

Предусловия:

    Существует сцена scene_id=5

    Существуют датасеты dataset_id=10 и dataset_id=11

Шаги выполнения:

| № | Действие                                       | Ожидаемый результат                      |
| - | ---------------------------------------------- | ---------------------------------------- |
| 1 | Добавить scene_id=5 в датасет 10 с split=train | Статус 200, сцена добавлена              |
| 2 | Добавить scene_id=5 в датасет 11 с split=val   | Статус 200, сцена добавлена без ошибки   |
| 3 | Проверить, что сцена не продублирована в БД    | GET /api/scenes/5 возвращает одну запись |

2.4. Модуль: INFER — Инференс и визуализация

TC-INFER-001: Успешный инференс на одной сцене

| Поле                 | Описание                                                   |
| -------------------- | ---------------------------------------------------------- |
| ID                   | TC-INFER-001                                               |
| Название             | Запуск и получение результата инференса на одной сцене ДЗЗ |
| Приоритет            | Критический                                                |
| Тип                  | Функциональное, позитивное                                 |
| Связь с требованиями | ФТ-INFER-001: Инференс сцены ДЗЗ                           |

Предусловия:

    Модель model_id=2 (обученная, status=active) доступна

    Сцена scene_id=7 (512×512, 4 канала) загружена в систему

    Пользователь авторизован

Тестовые данные:

```
model_id: 2
scene_id: 7
tile_size: 256
overlap: 32
```
Шаги выполнения:

| № | Действие                                                    | Ожидаемый результат                                     |
| - | ----------------------------------------------------------- | ------------------------------------------------------- |
| 1 | Отправить POST /api/inference/runs с model_id=2, scene_id=7 | Статус 201 Created, run_id возвращён                    |
| 2 | Получить GET /api/inference/runs/{run_id}                   | status=running или status=completed                     |
| 3 | Дождаться status=completed (не более 30 сек)                | Статус сменился на completed                            |
| 4 | Получить GET /api/inference/results?run_id={run_id}         | Возвращён объект с mask_uri, inference_time_sec         |
| 5 | Скачать маску по mask_uri                                   | Файл скачан, формат PNG/GeoTIFF, размер 512×512         |
| 6 | Визуально проверить маску                                   | Маска содержит значения классов от 0 до N-1 (не пустая) |

Постусловия:

    Результат сохранён в таблице inference_results

    Файл маски доступен по mask_uri

TC-INFER-002: Инференс на сцене с известной разметкой — расчёт mIoU

| Поле                 | Описание                                                            |
| -------------------- | ------------------------------------------------------------------- |
| ID                   | TC-INFER-002                                                        |
| Название             | Проверка расчёта метрик качества при инференсе на размеченной сцене |
| Приоритет            | Высокий                                                             |
| Тип                  | Функциональное, позитивное                                          |
| Связь с требованиями | ФТ-INFER-002: Расчёт метрик качества на тестовом наборе             |

Предусловия:

    Сцена scene_id=8 имеет привязанную разметку (label_id=15)

    Модель model_id=2 обучена

Шаги выполнения:

| № | Действие                                                      | Ожидаемый результат                                    |
| - | ------------------------------------------------------------- | ------------------------------------------------------ |
| 1 | Запустить инференс: POST /api/inference/runs с scene_id=8     | run_id возвращён                                       |
| 2 | Дождаться завершения                                          | status=completed                                       |
| 3 | Получить результат GET /api/inference/results?run_id={run_id} | Поля mean_iou и mean_f1 заполнены числовыми значениями |
| 4 | Проверить, что mean_iou ≥ 0.75                                | Значение соответствует целевой метрике модели          |
| 5 | Проверить mean_f1 > 0                                         | F1-score рассчитан корректно                           |

TC-INFER-003: Попытка инференса с несуществующей моделью

| Поле                 | Описание                                                        |
| -------------------- | --------------------------------------------------------------- |
| ID                   | TC-INFER-003                                                    |
| Название             | Отклонение запроса инференса при указании несуществующей модели |
| Приоритет            | Высокий                                                         |
| Тип                  | Функциональное, негативное                                      |
| Связь с требованиями | ФТ-INFER-001: Валидация входных данных инференса                |

Шаги выполнения:

| № | Действие                                           | Ожидаемый результат                          |
| - | -------------------------------------------------- | -------------------------------------------- |
| 1 | Отправить POST /api/inference/runs с model_id=9999 | Статус 404 Not Found                         |
| 2 | Проверить тело ответа                              | Сообщение: «Модель с ID 9999 не найдена»     |
| 3 | Проверить, что run не создан                       | GET /api/inference/runs — запись отсутствует |

TC-INFER-004: Пакетный инференс на датасете

| Поле                 | Описание                                                     |
| -------------------- | ------------------------------------------------------------ |
| ID                   | TC-INFER-004                                                 |
| Название             | Инференс по всем сценам тестового датасета пакетным запуском |
| Приоритет            | Средний                                                      |
| Тип                  | Функциональное, позитивное                                   |
| Связь с требованиями | ФТ-INFER-003: Пакетный инференс на датасете                  |

Предусловия:

    Датасет dataset_id=30 содержит 5 тестовых сцен

    Модель model_id=2 доступна

Шаги выполнения:

| № | Действие                                                               | Ожидаемый результат                                      |
| - | ---------------------------------------------------------------------- | -------------------------------------------------------- |
| 1 | Отправить POST /api/inference/runs с model_id=2, dataset_id=30         | Статус 201 Created                                       |
| 2 | Дождаться завершения всего запуска                                     | status=completed                                         |
| 3 | Получить список результатов GET /api/inference/results?run_id={run_id} | Возвращён список из 5 записей (по одной на каждую сцену) |
| 4 | Проверить, что каждая запись содержит mask_uri                         | Все маски доступны                                       |

TC-INFER-005: Проверка переходов статуса запуска инференса

| Поле                 | Описание                                                    |
| -------------------- | ----------------------------------------------------------- |
| ID                   | TC-INFER-005                                                |
| Название             | Корректность смены статуса InferenceRun по жизненному циклу |
| Приоритет            | Средний                                                     |
| Тип                  | Функциональное, бизнес-логика                               |
| Связь с требованиями | ФТ-INFER-001: Жизненный цикл запуска инференса              |

Workflow допустимых переходов:

text
QUEUED → RUNNING → COMPLETED
                 → FAILED
QUEUED → CANCELLED (пользователем)

Сценарий A — нормальное завершение:

| № | Действие                        | Ожидаемый результат |
| - | ------------------------------- | ------------------- |
| 1 | Создать запуск инференса        | status=queued       |
| 2 | Через 2 секунды получить статус | status=running      |
| 3 | Дождаться завершения            | status=completed    |

Сценарий B — отмена пользователем:

| № | Действие                                      | Ожидаемый результат                        |
| - | --------------------------------------------- | ------------------------------------------ |
| 1 | Создать запуск, получить run_id               | status=queued или running                  |
| 2 | Отправить DELETE /api/inference/runs/{run_id} | Статус 200, status=cancelled               |
| 3 | Попытаться снова изменить статус на completed | Статус 409: «Недопустимый переход статуса» |

## 3. НЕФУНКЦИОНАЛЬНЫЕ ТЕСТ-КЕЙСЫ

### 3.1. Тестирование производительности

TC-PERF-001: Время инференса одной сцены
| Поле                 | Описание                                              |
| -------------------- | ----------------------------------------------------- |
| ID                   | TC-PERF-001                                           |
| Название             | Проверка времени инференса сцены 1024×1024 пикселей   |
| Приоритет            | Высокий                                               |
| Тип                  | Нефункциональное, производительность                  |
| Связь с требованиями | НФТ-П-001: Время инференса ≤ 5 сек на сцену 1024×1024 |

Предусловия:

    GPU-узел с 1× NVIDIA A100 40GB активен

    Модель model_id=2 загружена в память

    10 тестовых сцен размером 1024×1024, 4 канала

Шаги выполнения:

| № | Действие                                               | Ожидаемый результат       |
| - | ------------------------------------------------------ | ------------------------- |
| 1 | Последовательно запустить инференс для 10 сцен         | Каждый запуск завершён    |
| 2 | Зафиксировать inference_time_sec из каждого результата | 10 значений зафиксированы |
| 3 | Вычислить среднее время                                | Среднее ≤ 3 сек           |
| 4 | Проверить максимальное время                           | Максимум ≤ 5 сек          |
| 5 | Вычислить 95-й перцентиль                              | P95 ≤ 4 сек               |

Инструменты: Locust (нагрузочный), inference_time_sec из API.

Метрики:

| Метрика         | Целевое значение |
| --------------- | ---------------- |
| Среднее время   | ≤ 3 сек          |
| 95-й перцентиль | ≤ 4 сек          |
| Максимум        | ≤ 5 сек          |

TC-PERF-002: Одновременный запуск 5 инференсов

| Поле                 | Описание                                                         |
| -------------------- | ---------------------------------------------------------------- |
| ID                   | TC-PERF-002                                                      |
| Название             | Производительность системы при 5 параллельных запросах инференса |
| Приоритет            | Средний                                                          |
| Тип                  | Нефункциональное, нагрузочное                                    |
| Связь с требованиями | НФТ-П-002: Поддержка минимум 5 одновременных сессий инференса    |

Шаги выполнения:

| № | Действие                                                                   | Ожидаемый результат                                                        |
| - | -------------------------------------------------------------------------- | -------------------------------------------------------------------------- |
| 1 | Отправить 5 запросов POST /api/inference/runs одновременно (5 разных сцен) | Все 5 запросов приняты, статус 201                                         |
| 2 | Через 30 секунд получить статус всех 5 запусков                            | Все 5 в статусе running или completed                                      |
| 3 | Дождаться завершения всех                                                  | Все 5 — completed, ни один — failed                                        |
| 4 | Проверить суммарное время                                                  | Время завершения последнего ≤ 25 сек (5 × 5 сек с параллельной обработкой) |

Инструменты: Locust, скрипт параллельных запросов на Python (asyncio + aiohttp).

### 3.2. Тестирование безопасности

TC-SEC-001: Защита API от SQL-инъекций
| Поле                 | Описание                                                      |
| -------------------- | ------------------------------------------------------------- |
| ID                   | TC-SEC-001                                                    |
| Название             | Проверка защиты REST API от SQL-инъекций в параметрах запроса |
| Приоритет            | Критический                                                   |
| Тип                  | Нефункциональное, безопасность                                |
| Связь с требованиями | НФТ-Б-003: Защита от SQL-инъекций                             |

Предусловия:

    Система развёрнута на тестовом стенде

    Инструмент OWASP ZAP настроен на тестовый endpoint

Тестовые данные (вредоносные):

text
' OR '1'='1
'; DROP TABLE users; --
1 UNION SELECT password FROM users --

Шаги выполнения:

| № | Действие                                     | Ожидаемый результат                                      |
| - | -------------------------------------------- | -------------------------------------------------------- |
| 1 | Отправить GET /api/scenes?source=' OR '1'='1 | Статус 400 или 200 с пустым списком, но НЕ утечка данных |
| 2 | Проверить логи сервера                       | Отсутствуют SQL-ошибки выполнения                        |
| 3 | Повторить с '; DROP TABLE users; --          | Таблица users существует, данные не потеряны             |
| 4 | Запустить автосканирование OWASP ZAP         | Отчёт ZAP: нет уязвимостей уровня High/Critical          |

Критерии прохождения:

    Ни одна из инъекций не выполнена на уровне БД

    Таблицы не повреждены

    OWASP ZAP не обнаружил SQLi-уязвимостей

TC-SEC-002: Горизонтальное разграничение данных между пользователями

| Поле                 | Описание                                                      |
| -------------------- | ------------------------------------------------------------- |
| ID                   | TC-SEC-002                                                    |
| Название             | Пользователь А не видит данные пользователя Б (IDOR-проверка) |
| Приоритет            | Критический                                                   |
| Тип                  | Нефункциональное, безопасность                                |
| Связь с требованиями | НФТ-Б-002: Изоляция данных пользователей                      |

Предусловия:

    Пользователь А (user_id=1) создал датасет dataset_id=100

    Пользователь Б (user_id=2) авторизован, токен получен

Шаги выполнения:

| № | Действие                                          | Ожидаемый результат                                       |
| - | ------------------------------------------------- | --------------------------------------------------------- |
| 1 | От имени пользователя Б: GET /api/datasets/100    | Статус 403 Forbidden или 404 Not Found                    |
| 2 | От имени пользователя Б: DELETE /api/datasets/100 | Статус 403 Forbidden                                      |
| 3 | Проверить, что датасет не удалён                  | GET /api/datasets/100 от имени пользователя А: статус 200 |

### 3.3. Тестирование юзабилити

TC-UX-001: Отображение прогресса обучения в реальном времени
| Поле                 | Описание                                                         |
| -------------------- | ---------------------------------------------------------------- |
| ID                   | TC-UX-001                                                        |
| Название             | Отображение актуальных метрик обучения без перезагрузки страницы |
| Приоритет            | Средний                                                          |
| Тип                  | Нефункциональное, юзабилити                                      |
| Связь с требованиями | НФТ-UX-001: Обновление метрик в реальном времени                 |

Предусловия:

    Эксперимент запущен, status=running

    Пользователь открыл страницу мониторинга эксперимента в браузере

Шаги выполнения:

| № | Действие                                                       | Ожидаемый результат                                                   |
| - | -------------------------------------------------------------- | --------------------------------------------------------------------- |
| 1 | Открыть страницу эксперимента                                  | Отображается график метрик loss и iou                                 |
| 2 | Подождать 60 секунд без перезагрузки страницы                  | График автоматически обновился — добавились новые точки               |
| 3 | Проверить, что последнее значение train_loss соответствует API | GET /api/experiments/{id}/metrics возвращает то же последнее значение |
| 4 | Проверить время обновления UI                                  | Задержка отображения новой метрики ≤ 10 секунд от появления в API     |

## 4. МАТРИЦА ТРАССИРУЕМОСТИ

| ID требования | Описание требования                       | Приоритет   | Тест-кейсы                               | Покрытие | Статус  |
| ------------- | ----------------------------------------- | ----------- | ---------------------------------------- | -------- | ------- |
| ФТ-AUTH-001   | Авторизация пользователя                  | Критический | TC-AUTH-001, TC-AUTH-002, TC-AUTH-003    | 100%     | Покрыто |
| НФТ-Б-001     | Защита от несанкционированного доступа    | Критический | TC-AUTH-003, TC-SEC-001                  | 100%     | Покрыто |
| НФТ-Б-002     | Разграничение доступа по ролям            | Критический | TC-AUTH-004, TC-SEC-002                  | 100%     | Покрыто |
| НФТ-Б-003     | Защита от SQL-инъекций                    | Критический | TC-SEC-001                               | 100%     | Покрыто |
| ФТ-DATA-001   | Импорт сцен ДЗЗ                           | Критический | TC-DATA-001, TC-DATA-002                 | 100%     | Покрыто |
| ФТ-DATA-002   | Загрузка разметки сцен                    | Высокий     | TC-DATA-003                              | 100%     | Покрыто |
| ФТ-DS-001     | Формирование обучающего датасета          | Критический | TC-DATASET-001, TC-DATASET-002           | 100%     | Покрыто |
| ФТ-TRAIN-001  | Самоконтролируемое предобучение           | Критический | TC-TRAIN-001, TC-TRAIN-003               | 100%     | Покрыто |
| ФТ-TRAIN-002  | Supervised-обучение с SSL-чекпоинтом      | Критический | TC-TRAIN-002                             | 100%     | Покрыто |
| ФТ-TRAIN-003  | Мониторинг метрик обучения                | Высокий     | TC-TRAIN-004                             | 100%     | Покрыто |
| ФТ-INFER-001  | Инференс сцены ДЗЗ                        | Критический | TC-INFER-001, TC-INFER-003, TC-INFER-005 | 100%     | Покрыто |
| ФТ-INFER-002  | Расчёт метрик качества                    | Высокий     | TC-INFER-002                             | 100%     | Покрыто |
| ФТ-INFER-003  | Пакетный инференс на датасете             | Средний     | TC-INFER-004                             | 100%     | Покрыто |
| НФТ-П-001     | Время инференса ≤ 5 сек / сцена 1024×1024 | Высокий     | TC-PERF-001                              | 100%     | Покрыто |
| НФТ-П-002     | Поддержка 5 параллельных сессий инференса | Средний     | TC-PERF-002                              | 100%     | Покрыто |
| НФТ-UX-001    | Обновление метрик в реальном времени      | Средний     | TC-UX-001                                | 100%     | Покрыто |



Сводная статистика:

    Всего требований: 16

    Требований «Критический»: 8

    Требований «Высокий»: 5

    Покрытие критических требований: 100%

    Покрытие высокоприоритетных: 100%

    Общее покрытие: 100%

```
Критический: ████████████████████ 100% (8/8)

Высокий:     ████████████████████ 100% (5/5)

Средний:     ████████████████████ 100% (3/3)

## 5. ПЛАН ВЫПОЛНЕНИЯ ТЕСТИРОВАНИЯ

| Этап                                              | Начало     | Окончание  | Ответственный |
| ------------------------------------------------- | ---------- | ---------- | ------------- |
| Подготовка тест-кейсов и тестовых данных          | 01.05.2027 | 05.05.2027 | ML-инженер    |
| Smoke-тестирование (AUTH, DATA)                   | 06.05.2027 | 06.05.2027 | ML-инженер    |
| Функциональное тестирование (AUTH, DATA, DATASET) | 09.05.2027 | 13.05.2027 | ML-инженер    |
| Функциональное тестирование (TRAIN, INFER)        | 16.05.2027 | 20.05.2027 | ML-инженер    |
| Нефункциональное тестирование (PERF, SEC, UX)     | 23.05.2027 | 27.05.2027 | ML-инженер    |
| Регрессионное тестирование                        | 28.05.2027 | 30.05.2027 | ML-инженер    |

## 6. КРИТЕРИИ ПРОХОЖДЕНИЯ ТЕСТИРОВАНИЯ

Система считается готовой к релизу при:

    Все 8 тест-кейсов с приоритетом «Критический» имеют статус Passed

    Pass Rate по всем тест-кейсам ≥ 98%

    Покрытие требований «Критический» и «Высокий» — 100%

    Отсутствуют открытые дефекты уровня Critical и Major

    Метрика mIoU на тестовом наборе ≥ 0.75 подтверждена тест-кейсом TC-INFER-002

    Время инференса соответствует TC-PERF-001: среднее ≤ 3 сек, максимум ≤ 5 сек

    Тест безопасности TC-SEC-001 пройден: OWASP ZAP не обнаружил уязвимостей уровня High/Critical

## ПРИЛОЖЕНИЯ

### Приложение А. Тестовые данные

Сцены для тестирования: 3 GeoTIFF-файла Sentinel-2 (512×512 и 1024×1024, 4 канала, 10 м/пкс)

Маски разметки: PNG-файлы с 5 классами (0=фон, 1=вода, 2=дороги, 3=здания, 4=растительность)

Конфигурации обучения: ssl_config_v1.yaml, supervised_config_v1.yaml (эпох: 3–5 для тестовых прогонов)

### Приложение Б. Конфигурация тестового окружения

ОС: Ubuntu 22.04 LTS

GPU: 1× NVIDIA A100 40GB

Python: 3.11

Фреймворк: PyTorch 2.2

СУБД: PostgreSQL 16

Объектное хранилище: MinIO (локальный S3-совместимый)

Тестовый URL API: http://localhost:8000/api